In [6]:
import cobra
import re
import json
import sys
import os
from cameo import models
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from networkx.algorithms.community.modularity_max import greedy_modularity_communities
from scipy import stats
from cobra import Model, Reaction, Metabolite


In [2]:
m = models.bigg.iML1515

# gene dictionaries for mapping between ID and names
gene_name_dic = {}
for g in m.genes:
    gene_name_dic[g.name] = g.id
    
gene_name_dic2 = {}
for g in m.genes:
    gene_name_dic2[g.id] = g.name

In [3]:
DES = pd.read_csv("E:\\1Mycount\\1111结果-总\\代谢模型\\gene.csv",header=None,index_col=0)
genes_ALL = list(DES.index)

In [4]:

m.reactions.EX_glc__D_e.bounds = (-10.0, 1000.0)
m.reactions.EX_o2_e.bounds = (-18.5, 1000.0)


        
for reaction in m.reactions:
    if reaction.bounds != (0.0, 0.0):
        if reaction.reversibility:
            reaction.bounds = (-1, 1)
        else:
            reaction.bounds = (0, 1)

In [9]:
metabolites = []
for met in m.metabolites:
    metabolites.append(met.id)

wt_yields = {}     
with m:
    for met in metabolites:
        #print(met)    逐个代谢物添加一个虚拟的排放反应，然后优化该反应，计算它的最大通量
        dm_id = str(met) + '_dummy_drain'
        DM_met = Reaction(dm_id)
        DM_met.name = met + ' dummy drain'
        DM_met.lower_bound = 0.0
        DM_met.upper_bound = 0.0
        DM_met.add_metabolites({m.metabolites.get_by_id(met) : -1.0})
        m.add_reactions([DM_met])

        m.reactions.get_by_id(dm_id).bounds = (0.0, 1000.0)
        m.objective = m.reactions.get_by_id(dm_id)           
        # set direction
        m.objective_direction = 'max'
        try:
            sol = m.optimize()
            #print(sol[dm_id])
            if sol.status == 'optimal':
                if sol[dm_id] > 0.00001:
                    wt_yields[dm_id] = sol[dm_id]
                else:
                    wt_yields[dm_id] = 0.0

            else:
                wt_yields[dm_id] = 0.0
        except:
            wt_yields[dm_id] = 0.0


MetYields_all = pd.DataFrame(index = metabolites)
MetYields_all.insert(0, column = 'wt', value = wt_yields.values())

MetYields_changes_all = pd.DataFrame(index = metabolites)
MetYields_changes_all.insert(0, column = 'wt', value = [1]*len(wt_yields.keys()))

count = 1
for gene in genes_ALL:

    ko_yields = []
    ko_changes = []
    with m:
        m.genes.get_by_id(gene).knock_out()
        for met in metabolites:
            
            # add drain reaction for metabolite
            dm_id = str(met) + '_dummy_drain'
            DM_met = Reaction(dm_id)
            DM_met.name = met + ' dummy drain'
            DM_met.lower_bound = 0.0
            DM_met.upper_bound = 0.0
            DM_met.add_metabolites({m.metabolites.get_by_id(met) : -1.0})
            m.add_reactions([DM_met])

            m.reactions.get_by_id(dm_id).bounds = (0.0, 1000.0)
            m.objective = m.reactions.get_by_id(dm_id)   # set the maximisation of metabolite production flux as objective        
            # set direction
            m.objective_direction = 'max'
            try:
                sol = m.optimize()
                if sol.status == 'optimal':
                    if sol[dm_id] > 0.00001:
                        ko_yields.append(sol[dm_id])
                        ko_changes.append(sol[dm_id]/wt_yields[dm_id])
                    else:
                        ko_yields.append(0.0)
                        ko_changes.append(0.0)
                else:
                    ko_yields.append(0.0)
                    ko_changes.append(0.0)
            except:
                ko_yields.append(0.0)
                ko_changes.append(0.0)
    MetYields_all.insert(count, column = gene, value = ko_yields) # add the yields of metabolites to dataframe
    MetYields_changes_all.insert(count, column = gene, value = ko_changes) # add change in metabolite yield compared to WT metabolite yield
    count += 1

MetYields_all.to_pickle('E:\\1Mycount\\1111结果-总\\代谢模型\\MetYields.pkl', protocol = 2)
MetYields_changes_all.to_pickle('E:\\1Mycount\\1111结果-总\\代谢模型\\MetYields_changes.pkl', protocol = 2)

In [11]:
MetYields = pd.read_pickle('E:\\1Mycount\\1111结果-总\\代谢模型\\MetYields.pkl')
MetYields289 = pd.DataFrame(index = MetYields.index)
count = 0
MetYields289.insert(0, column = 'wt', value = list(MetYields['wt'].values))
for ind, i in enumerate(MetYields.columns):
    if i in genes_ALL:
        count += 1
        MetYields289.insert(count, column = i, value = list(MetYields[i].values))

In [12]:
edges = []
for ind, g in enumerate(MetYields289.columns):
    if g != 'wt':
        for ind_met, met in enumerate(MetYields289.index):
            if MetYields289['wt'][ind_met] > 0.0001:
                if MetYields289[g][ind_met]/MetYields289['wt'][ind_met] < 0.00001:
                        
                    if '_c' in met:
                        met_new = met.split('_c')[0]
                    elif '_e' in met:
                        met_new = met.split('_e')[0]
                    elif '_p' in met:
                        met_new = met.split('_p')[0]
                    #if (m.genes.get_by_id(g).name, met_new) not in edges:
                    edges.append((m.genes.get_by_id(g).name, met_new))


# turn into graph
nodes1 = []
nodes2 = []
for edge in edges:
    nodes1.append(edge[0])
    nodes2.append(edge[1])         
nodes1 = list(set(nodes1))
nodes2 = list(set(nodes2))

G = nx.Graph()
G.add_nodes_from(nodes1+nodes2)
G.add_edges_from(edges)
print('Number of gene nodes:', len(nodes1))
print('Number of metabolite nodes:', len(nodes2))

Number of gene nodes: 8
Number of metabolite nodes: 109


In [13]:
c = list(greedy_modularity_communities(G))
# add labels to the nodes
gene_name_dic3 = {}
for g in m.genes:
    gene_name_dic3[g.name] = g.name
nx.set_node_attributes(G, gene_name_dic3, name = 'labels')


# colour code each metabolite in the network according to the cluster they have been assigned to
for met in nodes2:
    for ind, cluster in enumerate(c):
        if met in c[ind]:
            G.add_node(met, color = str(ind))

for gene in nodes1:
    for ind, cluster in enumerate(c):
        if gene in c[ind]:
            G.add_node(gene, color = str(ind))
                
nx.write_gml(G, 'E:\\1Mycount\\1111结果-总\\代谢模型\\Bipartite_MetaboliteYields_bothdir.gml')

In [14]:
def create_met_reacs_dataframe(nodes2, m):
    met_reacs = {}
    for i in nodes2:        
        if i + '_c' in m.metabolites:
            met_name = i + '_c'
            col = {}
            for r in m.reactions:
                if r in m.metabolites.get_by_id(met_name).reactions:
                    col[r.id] = 1
                else:
                    col[r.id] = 0
            met_reacs[i] = col
                
        elif i + '_p' in m.metabolites:
            met_name = i + '_p'
            col = {}
            for r in m.reactions:
                if r in m.metabolites.get_by_id(met_name).reactions:
                    col[r.id] = 1
                else:
                    col[r.id] = 0
            met_reacs[i] = col
            
            
                    
        
    df_met_reacs = pd.DataFrame.from_dict(met_reacs)
    
    return met_reacs, df_met_reacs

In [15]:
subsystems = []
for s in m.reactions:
    subsystems.append(s.subsystem)
subsystems = list(set(subsystems))  


reacs = []
for r in m.reactions:
    reacs.append(r.id)
df_reacs_sss_all = pd.DataFrame(index = reacs)

count = 0
for s in subsystems:
 
    s_genes = []
    for r in m.reactions:
        r_s = 0        
        if r.subsystem in s:

            r_s = 1
        s_genes.append(r_s)
    
    df_reacs_sss_all.insert(count, column = s, value = s_genes)
    count += 1

In [16]:
metabolites = []
for i in m.metabolites:
    if '_c' in i.id:
        met_id = i.id.split('_c')[0]
    elif '_e' in i.id:
        met_id = i.id.split('_e')[0]
    elif '_p' in i.id:
        met_id = i.id.split('_p')[0]
    metabolites.append(met_id)
metabolites = list(set(metabolites))   

# get the number of reactions associated to each cluster for each metabolic system
met_reacs, df_met_reacs = create_met_reacs_dataframe(metabolites, m)
df_met_sss = df_met_reacs.T.dot(df_reacs_sss_all)
df_met_sss[df_met_sss> 0] = 1

In [19]:
sig_pathways_clusters40 = {}
for ind, cluster_id in enumerate(c):
    genes = []
    genes_pathway = {}
    if len(c[ind]) > 5.0:
        print('')
        print(ind)
        nodes_list = []
        for i in c[ind]:
            if i in nodes2:
                nodes_list.append(i)


        met_reacs, df_met_reacs_c = create_met_reacs_dataframe(nodes_list, m)
        df_met_sss_c = df_met_reacs_c.T.dot(df_reacs_sss_all)
        df_met_sss_c[df_met_sss_c> 0] = 1


        p_value = []
        p_val_name = []
        for s in subsystems:
            x = df_met_sss_c.sum(axis = 0)[s]
            M = df_met_sss.sum(axis = 0)[s] # from all metabolites in the model
            if M != 0.0 and x != 0.0:
                p = stats.hypergeom.sf(x, len(metabolites), M, len(nodes_list))
            else:
                p = 1.0
                
            genes_path = []
            for ind_met, met in enumerate(df_met_sss_c[s]):
                if met > 0:
                    metabolite = df_met_sss_c.index[ind_met]
                    genes = G.neighbors(metabolite)
                    for ge in genes:
                        if ge in c[ind]:
                            genes_path.append(ge)
            genes_pathway[s] = list(set(genes_path))


                #if p < 0.05:
                #    print(ind, ss, x, M, p)


            p_value.append(p)
            p_val_name.append(s)
        print('')       
        print('corrected significance:')       


        from operator import itemgetter
        indices, pval_sorted = zip(*sorted(enumerate(p_value), key=itemgetter(1)))
        sig_pathways_clusters40[ind] = {}
        sig_pathways = []
        for indj, i in enumerate(pval_sorted):
            if i < ((indj+1)/40)*0.01:
                #print(p_val_name[indices[ind]], i, ((ind+1)/40)*0.01)
                sig_pathways.append(p_val_name[indices[indj]])
                sig_pathways_clusters40[ind].update({p_val_name[indices[indj]] : genes_pathway[p_val_name[indices[indj]]]})
              #  print(p_val_name[indices[indj]], i, ((ind+1)/40)*0.01)
                print(f"{p_val_name[indices[indj]]}, p = {i}, FDR = {((indj+1)/40)*0.01}")
              #  print(genes_pathway[p_val_name[indices[indj]]])
        if sig_pathways == []:
                sig_pathways_clusters40[ind] = {}


0

corrected significance:
Membrane Lipid Metabolism, p = 1.286005066460999e-31, FDR = 0.00025
Methionine Metabolism, p = 8.012125641924661e-07, FDR = 0.0005
Cofactor and Prosthetic Group Biosynthesis, p = 1.1035445884803995e-05, FDR = 0.00075
Cysteine Metabolism, p = 0.000387257679792366, FDR = 0.001
Arginine and Proline Metabolism, p = 0.0008114309695072939, FDR = 0.00125

1

corrected significance:
Inorganic Ion Transport and Metabolism, p = 1.1676225083725675e-07, FDR = 0.00025


In [22]:
met_pathways = pd.read_excel('E:\\1Mycount\\ML-GSM\\GSM\\date\\Met_Pathways1.xlsx', sheet_name='Sheet1')
met_pathways2 = pd.read_excel('E:\\1Mycount\\ML-GSM\\GSM\\date\\Met_Pathways1.xlsx', sheet_name='Sheet2')
metabolite_pathways = {}
for i in m.metabolites:
    pathways = []
    if 'biocyc' in i.annotation:
        for ind, met in enumerate(met_pathways['Metabolite name']):
            if i.annotation['biocyc'][0].split('META:')[1] == met:
                #print(i, met_pathways2['Pathways of compound'][ind])
                if str(met_pathways2['Pathways of compound'][ind]) != 'nan':
                    pathways = met_pathways2['Pathways of compound'][ind].split(' // ')
    metabolite_pathways[i.id] = pathways
pathways = []
for i, j in metabolite_pathways.items():
    for path in j:
        pathways.append(path)
        
pathways = list(set(pathways))
len(pathways)

436

In [23]:
sig_pathways_clusters = {}
for ind, cluster in enumerate(c):
    if len(c[ind]) > 5:
        print(ind)
        p_value = []
        p_val_name = []
        genes_pathway = {}
        for i in pathways:
            number_mets_ml = 0.0
            number_mets_model = 0.0
            genes_path = []
            for met, j in metabolite_pathways.items():
                for path in j:
                    if i == path:
                        met_id = met.split('_')[0]
                        if met_id in c[ind]:
                            number_mets_ml += 1.0
                            for ge in nodes1:
                                if (ge, met_id) in edges:
                                    if ge in c[ind]:
                                        genes_path.append(ge)

                        if met_id in metabolites:
                            number_mets_model += 1.0


            l_c = 0.0
            for met in c[ind]:
                if met in nodes2:
                    l_c += 1.0
            if number_mets_ml > 3.0:
                p = stats.hypergeom.sf(number_mets_ml, len(metabolites), number_mets_model, l_c)
                #if p < 0.05:
                #    print(i, p, number_mets_ml, number_mets_model)

                p_value.append(p)
                p_val_name.append(i)
                genes_pathway[i] = list(set(genes_path))
                #print(set(genes_path))

            #else:
            #    p_value.append(1.0)
            #    p_val_name.append(i)


        print('')       
        print('corrected significance:')       

        sig_pathways_clusters[ind] = {}
        from operator import itemgetter
        indices, pval_sorted = zip(*sorted(enumerate(p_value), key=itemgetter(1)))
        sig_pathways = []
        for indj, i in enumerate(pval_sorted):
            if i < ((indj+1)/436)*0.01:
                print(p_val_name[indices[indj]], ((indj+1)/396)*0.01)
                #print(((indj+1)/396)*0.01)
                sig_pathways_clusters[ind].update({p_val_name[indices[indj]] : genes_pathway[p_val_name[indices[indj]]]})
                print(genes_pathway[p_val_name[indices[indj]]])

0

corrected significance:
glutathione-glutaredoxin redox reactions 2.5252525252525256e-05
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
fatty acid &beta;-oxidation I (generic) 5.050505050505051e-05
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
oleate &beta;-oxidation 7.575757575757576e-05
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
CDP-diacylglycerol biosynthesis I 0.00010101010101010102
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
glutathionylspermidine biosynthesis 0.00012626262626262626
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
<i>S</i>-methyl-5'-thioadenosine degradation IV 0.00015151515151515152
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
biotin biosynthesis from 8-amino-7-oxononanoate I 0.00017676767676767677
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
L-cysteine biosynthesis VII (from <i>S</i>-sulfo-L-cysteine) 0.00020202020202020205
['cysD', 'cysC', 'cysJ', 'cysI', 'cysH', 'cysN']
superpathway of phospholipid biosynthesis I (bacteria) 0.00022727

ValueError: not enough values to unpack (expected 2, got 0)

In [24]:
grouped_sig_pathways = {}
grouped_sig_pathways['Proline metabolism'] = ['L-proline biosynthesis I (from L-glutamate)', 'Arginine and Proline Metabolism']
grouped_sig_pathways['Fatty acid degradation'] = ['fatty acid &beta;-oxidation I (generic)',
                                                 'oleate &beta;-oxidation']
grouped_sig_pathways['Purine metabolism'] = ['superpathway of purine nucleotides <i>de novo</i> biosynthesis II',
                                              'guanosine ribonucleotides <i>de novo</i> biosynthesis',
                                              'superpathway of guanosine nucleotides <i>de novo</i> biosynthesis II',
                                              'superpathway of histidine, purine, and pyrimidine biosynthesis',
                                              'superpathway of guanine and guanosine salvage', 
                                               'adenine and adenosine salvage III', 
                                               'guanine and guanosine salvage III',
                                               'guanine and guanosine salvage',
                                               'xanthine and xanthosine salvage',
                                               'adenine and adenosine salvage II',
                                               'adenine and adenosine salvage V',
                                               'guanosine deoxyribonucleotides <i>de novo</i> biosynthesis II',
                                               "inosine-5'-phosphate biosynthesis I",
                                               "purine ribonucleosides degradation",
                                             "adenosine nucleotides degradation II",
                                             "guanosine nucleotides degradation III"
                                             "purine deoxyribonucleosides degradation I",
                                             "superpathway of purine deoxyribonucleosides degradation",
                                             "superpathway of purine nucleotides <i>de novo</i> biosynthesis II",
                                             "adenosine ribonucleotides <i>de novo</i> biosynthesis",
                                             "purine deoxyribonucleosides degradation I",
                                             "guanosine nucleotides degradation III",
                                             "superpathway of adenosine nucleotides <i>de novo</i> biosynthesis II",
                                             "5-aminoimidazole ribonucleotide biosynthesis I",
                                             '5-aminoimidazole ribonucleotide biosynthesis II',
                                             'superpathway of 5-aminoimidazole ribonucleotide biosynthesis',
                                             "Nucleotide Salvage Pathway"
                                             
                                              ]


grouped_sig_pathways['Folate metabolism'] = ['superpathway of tetrahydrofolate biosynthesis', 
                                            '6-hydroxymethyl-dihydropterin diphosphate biosynthesis I',
                                            'folate transformations III (<i>E. coli</i>)',
                                            'tetrahydrofolate biosynthesis', 'folate polyglutamylation', 'Folate Metabolism']



grouped_sig_pathways['Energy carriers biosynthesis'] = ['NAD salvage pathway I', 
                                                        'flavin biosynthesis I (bacteria and plants)',
                                                       'NAD salvage pathway IV (from nicotinamide riboside)',
                                                       'NAD salvage pathway I (PNC VI cycle)',
                                                       '2-carboxy-1,4-naphthoquinol biosynthesis', 
                                                       'superpathway of ubiquinol-8 biosynthesis (early decarboxylation)',
                                                       'superpathway of menaquinol-8 biosynthesis I',
                                                       'glutathione-glutaredoxin redox reactions',
                                                       'ubiquinol-8 biosynthesis (early decarboxylation)', 'NAD salvage pathway II (PNC IV cycle)']

grouped_sig_pathways['Nucleic acid processing'] = ['tRNA processing', 'tRNA-uridine 2-thiolation and selenation (bacteria)',
                                                  'queuosine biosynthesis I (<i>de novo</i>)']


grouped_sig_pathways['Peptidoglycan metabolism'] = ['muropeptide degradation', 
                                                    'peptidoglycan biosynthesis I (<I>meso</I>-diaminopimelate containing)',
                                                    'UDP-<i>N</i>-acetylmuramoyl-pentapeptide biosynthesis I (<i>meso</i>-diaminopimelate containing)',
                                                    'anhydromuropeptides recycling I', 'peptidoglycan maturation (<i>meso</i>-diaminopimelate containing)', 'Murein Biosynthesis',
                                                    'Murein Recycling',
                                                    
                                                   ]

grouped_sig_pathways['Fatty acid and lipid biosynthesis'] = ['fatty acid biosynthesis initiation III', '(5Z)-dodecenoate biosynthesis I',
                                                            'fatty acid biosynthesis initiation (type II)', 'superpathway of unsaturated fatty acids biosynthesis (<i>E. coli</i>)',
                                                            'superpathway of fatty acid biosynthesis initiation (E. coli)', '(Kdo)<sub>2</sub>-lipid A biosynthesis I (E. coli)',
                                                             'fatty acid elongation -- saturated', 'palmitoleate biosynthesis I (from (5Z)-dodec-5-enoate)', 'fatty acid biosynthesis initiation II',
                                                            'superpathway of fatty acid biosynthesis I (E. coli)', '<i>cis</i>-vaccenate biosynthesis', 'palmitate biosynthesis (type II fatty acid synthase)', 'Membrane Lipid Metabolism']

grouped_sig_pathways['Pyrimidine metabolism'] = ['salvage pathways of pyrimidine deoxyribonucleotides', 
                                                'superpathway of pyrimidine ribonucleosides salvage',
                                                'superpathway of pyrimidine deoxyribonucleotides <i>de novo</i> biosynthesis',
                                                'salvage pathways of pyrimidine ribonucleotides',
                                                 'pyrimidine deoxyribonucleosides degradation',
                                                 'pyrimidine deoxyribonucleotide phosphorylation',
                                                 'pyrimidine deoxyribonucleotides <i>de novo</i> biosynthesis I',
                                                 'superpathway of pyrimidine deoxyribonucleosides degradation',
                                                 'pyrimidine ribonucleosides salvage I',
                                                 'superpathway of pyrimidine deoxyribonucleotides <i>de novo</i> biosynthesis (<i>E. coli</i>)',
                                                 'pyrimidine deoxyribonucleotides dephosphorylation',
                                                 'pyrimidine deoxyribonucleotides <i>de novo</i> biosynthesis II',
                                                 'pyrimidine nucleobases salvage I',
                                                 'pyrimidine ribonucleosides salvage III',
                                                 'pyrimidine ribonucleosides salvage II',
                                                 'pyrimidine nucleobases salvage II',
                                                 'CMP phosphorylation', 
                                                 'superpathway of pyrimidine ribonucleotides <i>de novo</i> biosynthesis',
                                                 'pyrimidine ribonucleosides degradation',
                                                 'superpathway of pyrimidine nucleobases salvage',
                                                 'UMP biosynthesis I',
                                                 'uracil degradation III',
                                                 'UTP and CTP <i>de novo</i> biosynthesis'
                                                                                                 ]

grouped_sig_pathways['Lipolysaccharides metabolism'] = ['superpathway of lipopolysaccharide biosynthesis',
                                                        'lipid A-core biosynthesis (<i>E. coli</i> K-12)',
                                                        'superpathway of (Kdo)<SUB>2</SUB>-lipid A biosynthesis',
                                                        'Kdo transfer to lipid IV<sub>A</sub> I (E. coli)',
                                                        'lipid IV<sub>A</sub> biosynthesis (E. coli)',
                                                        'polymyxin resistance', 'Lipopolysaccharide Biosynthesis / Recycling',
                                                       ]

grouped_sig_pathways['Enterobactin metabolism'] = [ 
                                                   'enterobactin biosynthesis']

grouped_sig_pathways['Carrier biosynthesis'] = ['acyl carrier protein metabolism']


grouped_sig_pathways['Heme metabolism'] = ['heme <i>b</i> biosynthesis II (oxygen-independent)', 
                                          'heme <i>b</i> biosynthesis V (aerobic)',
                                          'superpathway of heme <i>b</i> biosynthesis from uroporphyrinogen-III',
                                           'cytochrome <i>c</i> biogenesis (system I type)'
                                          ]
grouped_sig_pathways['Chorismate metabolism'] = ['superpathway of chorismate metabolism', 'chorismate biosynthesis from 3-dehydroquinate']

grouped_sig_pathways['Electron transport chain'] = ['hydrogen to fumarate electron transfer', 'NADH to fumarate electron transfer', 'NADH to hydrogen peroxide electron transfer']
grouped_sig_pathways['Phospholipid metabolism'] = ['superpathway of phospholipid biosynthesis I (bacteria)', 'CDP-diacylglycerol biosynthesis I', 'cardiolipin biosynthesis III', 'CDP-diacylglycerol biosynthesis II', 'Glycerophospholipid Metabolism',
                                                  'phosphatidylserine and phosphatidylethanolamine biosynthesis I',]
grouped_sig_pathways["Pyridoxal 5'-phosphate biosynthesis"] = ["superpathway of pyridoxal 5'-phosphate biosynthesis and salvage",
                                                              "pyridoxal 5'-phosphate salvage I"]

grouped_sig_pathways['Sugar nucleotide biosynthesis'] = ['UDP-&alpha;-D-glucose biosynthesis', 'UDP-&alpha;-D-galactose biosynthesis', 
                                                        'UDP-<i>N</i>-acetyl-&alpha;-D-mannosaminouronate biosynthesis',
                                                        'galactose degradation I (Leloir pathway)',
                                                        'dTDP-&beta;-L-rhamnose biosynthesis',
                                                        'UDP-&alpha;-D-glucuronate biosynthesis (from UDP-glucose)',
                                                         'dTDP-<i>N</i>-acetylthomosamine biosynthesis',
                                                        '<i>O</i>-antigen building blocks biosynthesis (<i>E. coli</i>)',
                                                        'ADP-L-<i>glycero</i>-&beta;-D-<i>manno</i>-heptose biosynthesis',
                                                         'CMP-3-deoxy-D-<I>manno</I>-octulosonate biosynthesis', 'GDP-L-fucose biosynthesis I (from GDP-D-mannose)',]
grouped_sig_pathways['Allantoin degradation'] = ['allantoin degradation IV (anaerobic)', 'allantoin degradation to glyoxylate III']
grouped_sig_pathways['putrescine degradation'] = ['putrescine degradation II']

grouped_sig_pathways['Sulfur compound metabolism'] = ['assimilatory sulfate reduction I', 'superpathway of sulfate assimilation and cysteine biosynthesis']

grouped_sig_pathways['Tetrapyrrole biosynthesis'] = ['tetrapyrrole biosynthesis I (from glutamate)']
grouped_sig_pathways['Biotin metabolism'] = ['biotin biosynthesis I', 'biotin biosynthesis from 8-amino-7-oxononanoate I']
grouped_sig_pathways['Autoinducer AI-2 biosynthesis'] = ['autoinducer AI-2 biosynthesis I']


grouped_sig_pathways['Tyrosine, Tryptophan, and Phenylalanine Metabolism'] = ['L-tyrosine biosynthesis I', 'L-tryptophan biosynthesis',]
grouped_sig_pathways['Cysteine metabolism'] = ['L-cysteine biosynthesis VII (from <i>S</i>-sulfo-L-cysteine)', 'Cysteine Metabolism', 'L-cysteine degradation II']
grouped_sig_pathways['Histidine metabolism'] = ['Histidine metabolism', 'L-histidine biosynthesis', 'Histidine Metabolism']

grouped_sig_pathways['Arginine metabolism'] = ['L-arginine degradation II (AST pathway)', 'L-arginine biosynthesis I (via L-ornithine)']
grouped_sig_pathways['Glycerol degradation'] = ['glycerophosphodiester degradation', 'glycerol and glycerophosphodiester degradation']
grouped_sig_pathways['Lipoate biosynthesis'] = ['lipoate biosynthesis', 'lipoate biosynthesis and incorporation I']
grouped_sig_pathways['Methionine metabolism'] = ['L-methionine biosynthesis I', 'Methionine Metabolism']
grouped_sig_pathways['Lysine and threonine metabolism'] = ['L-lysine biosynthesis I', 'Threonine and Lysine Metabolism',]
grouped_sig_pathways['Nitrogen metabolism'] = ['Nitrogen Metabolism']
grouped_sig_pathways['Methylerythritol phosphate pathway'] = ['methylerythritol phosphate pathway I']
grouped_sig_pathways['Oxidative phosphorylation'] = ['Oxidative Phosphorylation']
grouped_sig_pathways['Pentose phosphate pathway'] = ['Pentose Phosphate Pathway',]
grouped_sig_pathways['Cell envelope biosynthesis'] = ['Cell Envelope Biosynthesis',]
grouped_sig_pathways['Aspartate metabolism'] = ['aspartate superpathway',]
grouped_sig_pathways['tRNA charging'] = ['tRNA Charging']
grouped_sig_pathways['Aromatic amino acids biosynthesis'] = ['superpathway of aromatic amino acid biosynthesis']
grouped_sig_pathways['Colanic acid building blocks biosynthesis'] = ['colanic acid building blocks biosynthesis']
grouped_sig_pathways['S-adenosyl-L-methionine salvage I'] = ['<i>S</i>-adenosyl-L-methionine salvage I']
grouped_sig_pathways['Arsenate detoxification (glutaredoxin)'] = ['arsenate detoxification II (glutaredoxin)']
grouped_sig_pathways['Glycine and serine metabolism'] = ['glycine biosynthesis I', 'Glycine and Serine Metabolism','L-serine biosynthesis I', 'superpathway of L-serine and glycine biosynthesis I']
grouped_sig_pathways['Cofactor metabolism'] = ['Cofactor and Prosthetic Group Biosynthesis',]
grouped_sig_pathways['8-amino-7-oxononanoate biosynthesis'] = ['8-amino-7-oxononanoate biosynthesis I']
grouped_sig_pathways['PreQ0 biosynthesis'] = ['preQ<sub>0</sub> biosynthesis']
grouped_sig_pathways['Periplasmic disulfide bond formation'] = ['periplasmic disulfide bond formation']
grouped_sig_pathways['Tetrahydromonapterin biosynthesis'] = ['tetrahydromonapterin biosynthesis']
grouped_sig_pathways['Cobamide biosynthesis'] = ['superpathway of adenosylcobalamin salvage from cobinamide I']
grouped_sig_pathways['Leucine and Isoleucine Metabolism'] = ['L-isoleucine biosynthesis I (from threonine)','L-leucine biosynthesis', 'Valine, Leucine, and Isoleucine Metabolism']
grouped_sig_pathways['CoA biosynthesis'] = ['coenzyme A biosynthesis I (prokaryotic)','superpathway of coenzyme A biosynthesis I (bacteria)', 'superpathway of coenzyme A biosynthesis I (bacteria)']
grouped_sig_pathways['Polyisoprenoid biosynthesis'] = ['polyisoprenoid biosynthesis (<i>E. coli</i>)']
grouped_sig_pathways["S-methyl-5'-thioadenosine degradation"] = ["<i>S</i>-methyl-5'-thioadenosine degradation IV"]
grouped_sig_pathways['Trans, trans-farnesyl diphosphate biosynthesis'] = ['<i>trans, trans</i>-farnesyl diphosphate biosynthesis']
grouped_sig_pathways['Spermidine biosynthesis'] = ['spermidine biosynthesis I']
grouped_sig_pathways['Enterobacterial common antigen biosynthesis'] =['enterobacterial common antigen biosynthesis']
grouped_sig_pathways['Phosphopantothenate biosynthesis I'] = ['phosphopantothenate biosynthesis I']
grouped_sig_pathways['Glutathionylspermidine biosynthesis'] = ['glutathionylspermidine biosynthesis']
grouped_sig_pathways['Phosphopantothenate biosynthesis'] = ['phosphopantothenate biosynthesis I']
grouped_sig_pathways['Citrate lyase activation'] = ['citrate lyase activation']

In [25]:
sig_pathways = []
for ind, data in sig_pathways_clusters.items():
    for path, genes in data.items():
        match = 0
        for group, pathways in grouped_sig_pathways.items():
            if path in pathways:
                match = 1
                sig_pathways.append(group)
        if match == 0:
            sig_pathways.append(path)
sig_pathways = list(set(sig_pathways))




for ind, data in sig_pathways_clusters40.items():
    for path, genes in data.items():
        match = 0
        for group, pathways in grouped_sig_pathways.items():
            if path in pathways:
                match = 1
                sig_pathways.append(group)
        if match == 0:
            sig_pathways.append(path)
sig_pathways = list(set(sig_pathways))

In [26]:
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

    
# Biocyc sig.
edges_sp = []
for C_ind, data in sig_pathways_clusters.items():
    for pathway, genes in sig_pathways_clusters[C_ind].items():
        match = 0.0
        for group, p in grouped_sig_pathways.items():
            if pathway in p:
                path_new = group
                match = 1.0
            
        if match == 0.0:
            path_new = pathway

        for gene in genes:
            edges_sp.append((path_new, gene))
        
        

            
            
# Bigg pathways sig. 

for C_ind, data in sig_pathways_clusters40.items():
    for pathway, genes in sig_pathways_clusters40[C_ind].items():
        match = 0.0
        for group, p in grouped_sig_pathways.items():
            if pathway in p:
                path_new = group
                match = 1.0
            
        if match == 0.0:
            path_new = pathway

        for gene in genes:
            edges_sp.append((path_new, gene))
            
nodes1_sp = []# pathways
nodes2_sp = []# genes
for i, j in edges_sp:
    nodes1_sp.append(i)
    nodes2_sp.append(j)
    
nodes1_sp = list(set(nodes1_sp))
nodes2_sp = list(set(nodes2_sp))


G2 = nx.Graph()
G2.add_nodes_from(nodes1_sp+nodes2_sp)
G2.add_edges_from(edges_sp)



# colour code each gene in the network according to the cluster they have been assigned to
for gene in nodes2_sp:
    for ind, cluster in enumerate(c):
        if gene in c[ind]:
            G2.add_node(gene, color = str(ind))

                
nx.write_gml(G2, 'E:\\1Mycount\\1111结果-总\\代谢模型\\SigPathways_genes_MetYields_Greedy_bothdir.gml') 